# Refine

> Postprocessing markdown files including fixing heading hierarchy and adding image descriptions

In [ ]:
#| default_exp refine

This module aims to fix and enrich markdown headings from OCR'd PDF files by:

1. Fixing heading hierarchy that was corrupted during OCR
2. Adding page numbers to headings for better navigation
3. Enriching figure references with descriptive text and creating a table of figures

[TBD]

In [ ]:
#| export
from mistocr.core import read_pgs
from re import sub, findall, MULTILINE

# Dev only
import sys
sys.path.append('../../..')
from ctx_utils import *

In [ ]:
read_pgs??


```python
def read_pgs(
    path:str, # OCR output directory, 
    pg:int=None, # Page number
    ) -> str:
    "Read specific page or all pages from OCR output directory"
    path = Path(path)
    if pg:
        pg_path = path / f'page_{pg}.md'
        if not pg_path.exists(): raise ValueError(f"Page {pg} not found")
        return pg_path.read_text()
    pgs = sorted(path.glob('page_*.md'), key=lambda p: int(p.stem.split('_')[1]))
    return '\n\n'.join([p.read_text() for p in pgs])
```

**File:** `~/.local/lib/python3.12/site-packages/mistocr/core.py`

In [ ]:
md = read_pgs('files/test/md_all/resnet')
md[:500]

'# Deep Residual Learning for Image Recognition \n\nKaiming He Xiangyu Zhang Shaoqing Ren Jian Sun<br>Microsoft Research<br>\\{kahe, v-xiangz, v-shren, jiansun\\}@microsoft.com\n\n\n#### Abstract\n\nDeeper neural networks are more difficult to train. We present a residual learning framework to ease the training of networks that are substantially deeper than those used previously. We explicitly reformulate the layers as learning residual functions with reference to the layer inputs, instead of learning unr'

In [ ]:
from toolslm.md_hier import *

# Lisette Core

# AUTOGENERATED! DO NOT EDIT! File to edit: ../nbs/00_core.ipynb.

# %% auto 0
__all__ = ['sonn45', 'detls_tag', 're_tools', 'effort', 'patch_litellm', 'remove_cache_ckpts', 'mk_msg', 'fmt2hist', 'mk_msgs',
           'stream_with_complete', 'lite_mk_func', 'ToolResponse', 'cite_footnote', 'cite_footnotes', 'Chat',
           'random_tool_id', 'mk_tc', 'mk_tc_req', 'mk_tc_result', 'mk_tc_results', 'astream_with_complete',
           'AsyncChat', 'mk_tr_details', 'AsyncStreamFormatter', 'adisplay_stream']

# %% ../nbs/00_core.ipynb
import asyncio, base64, json, litellm, mimetypes, random, string
from typing import Optional
from html import escape
from litellm import (acompletion, completion, stream_chunk_builder, Message,
                     ModelResponse, ModelResponseStream, get_model_info, register_model, Usage)
from litellm.utils import function_to_dict, StreamingChoices, Delta, ChatCompletionMessageToolCall, Function, Choices
from toolslm.funccall import mk_ns, call_func, call_func_async, get_schema
from fastcore.utils import *
from fastcore import imghdr
from dataclasses import dataclass

# %% ../nbs/00_core.ipynb
def patch_litellm(seed=0):
    "Patch litellm.ModelResponseBase such that `id` and `created` are fixed."
    from litellm.types.utils import ModelResponseBase
    @patch
    def __init__(self: ModelResponseBase, id=None, created=None, *args, **kwargs): 
        self._orig___init__(id='chatcmpl-xxx', created=1000000000, *args, **kwargs)

    @patch
    def __setattr__(self: ModelResponseBase, name, value):
        if name == 'id': value = 'chatcmpl-xxx'
        elif name == 'created': value = 1000000000
        self._orig___setattr__(name, value)

    if seed is not None: random.seed(seed) # ensures random ids like tool call ids are deterministic

# %% ../nbs/00_core.ipynb
@patch
def _repr_markdown_(self: litellm.ModelResponse):
    message = self.choices[0].message
    content = ''
    if mc:=message.content: content += mc[0]['text'] if isinstance(mc,list) else mc
    if message.tool_calls:
        tool_calls = [f"\n\n🔧 {nested_idx(tc,'function','name')}({nested_idx(tc,'function','arguments')})\n" for tc in message.tool_calls]
        content += "\n".join(tool_calls)
    if not content: content = str(message)
    details = [
        f"id: `{self.id}`",
        f"model: `{self.model}`",
        f"finish_reason: `{self.choices[0].finish_reason}`"
    ]
    if hasattr(self, 'usage') and self.usage: details.append(f"usage: `{self.usage}`")
    det_str = '\n- '.join(details)
    
    return f"""{content}

<details>

- {det_str}

</details>"""

# %% ../nbs/00_core.ipynb
register_model({
    "claude-sonnet-4-5": {
        "max_tokens": 64000, "max_input_tokens": 200000, "max_output_tokens": 64000,
        "input_cost_per_token": 3e-06, "output_cost_per_token": 1.5e-05, "cache_creation_input_token_cost": 3.75e-06, "cache_read_input_token_cost": 3e-07,
        "litellm_provider": "anthropic", "mode": "chat",
        "supports_function_calling": True, "supports_parallel_function_calling": True, "supports_vision": True, "supports_prompt_caching": True, "supports_response_schema": True, "supports_system_messages": True, "supports_reasoning": True, "supports_assistant_prefill": True,
        "supports_tool_choice": True, "supports_computer_use": True 
    }
});
sonn45 = "claude-sonnet-4-5"

# %% ../nbs/00_core.ipynb
def _bytes2content(data):
    "Convert bytes to litellm content dict (image or pdf)"
    mtype = 'application/pdf' if data[:4] == b'%PDF' else mimetypes.types_map.get(f'.{imghdr.what(None, h=data)}')
    if not mtype: raise ValueError(f'Data must be image or PDF bytes, got {data[:10]}')
    return {'type': 'image_url', 'image_url': f'data:{mtype};base64,{base64.b64encode(data).decode("utf-8")}'}

# %% ../nbs/00_core.ipynb
def _add_cache_control(msg,          # LiteLLM formatted msg
                       ttl=None):    # Cache TTL: '5m' (default) or '1h'
    "cache `msg` with default time-to-live (ttl) of 5minutes ('5m'), but can be set to '1h'."
    if isinstance(msg["content"], str): 
        msg["content"] = [{"type": "text", "text": msg["content"]}]
    cache_control = {"type": "ephemeral"}
    if ttl is not None: cache_control["ttl"] = ttl
    if isinstance(msg["content"], list) and msg["content"]:
        msg["content"][-1]["cache_control"] = cache_control
    return msg

def _has_cache(msg):
    return msg["content"] and isinstance(msg["content"], list) and ('cache_control' in msg["content"][-1])

def remove_cache_ckpts(msg):
    "remove cache checkpoints and return msg."
    if _has_cache(msg): msg["content"][-1].pop('cache_control', None)
    return msg

def _mk_content(o):
    if isinstance(o, str): return {'type':'text','text':o.strip() or '.'}
    elif isinstance(o,bytes): return _bytes2content(o)
    return o

# %% ../nbs/00_core.ipynb
def mk_msg(
    content,      # Content: str, bytes (image), list of mixed content, or dict w 'role' and 'content' fields
    role="user",  # Message role if content isn't already a dict/Message
    cache=False,  # Enable Anthropic caching
    ttl=None      # Cache TTL: '5m' (default) or '1h'
):
    "Create a LiteLLM compatible message."
    if isinstance(content, dict) or isinstance(content, Message): return content
    if isinstance(content, ModelResponse): return content.choices[0].message
    if isinstance(content, list) and len(content) == 1 and isinstance(content[0], str): c = content[0]
    elif isinstance(content, list): c = [_mk_content(o) for o in content]
    else: c = content
    msg = {"role": role, "content": c}
    return _add_cache_control(msg, ttl=ttl) if cache else msg

# %% ../nbs/00_core.ipynb
detls_tag = "<details class='tool-usage-details'>"
re_tools = re.compile(fr"^({detls_tag}\n+```json\n+(.*?)\n+```\n+</details>)", flags=re.DOTALL|re.MULTILINE)

# %% ../nbs/00_core.ipynb
def _extract_tool(text:str)->tuple[dict,dict]:
    "Extract tool call and results from <details> block"
    d = json.loads(text.strip())
    call = d['call']
    func = call['function']
    tc = ChatCompletionMessageToolCall(Function(dumps(call['arguments']),func), d['id'])
    tr = {'role': 'tool','tool_call_id': d['id'],'name': func, 'content': d['result']}
    return tc,tr

def fmt2hist(outp:str)->list:
    "Transform a formatted output into a LiteLLM compatible history"
    lm,hist = Message(),[]
    spt = re_tools.split(outp)
    for txt,_,tooljson in chunked(spt, 3, pad=True):
        txt = txt.strip() if tooljson or txt.strip() else '.'
        hist.append(lm:=Message(txt))
        if tooljson:
            tcr = _extract_tool(tooljson)
            if not hist: hist.append(lm) # if LLM calls a tool without talking
            lm.tool_calls = lm.tool_calls+[tcr[0]] if lm.tool_calls else [tcr[0]] 
            hist.append(tcr[1])
    return hist

# %% ../nbs/00_core.ipynb
def _apply_cache_idxs(msgs, cache_idxs=[-1], ttl=None):
    'Add cache control to idxs after filtering tools'
    ms = L(msgs).filter(lambda m: not (m.get('tool_calls', []) or m['role'] == 'tool'))
    for i in cache_idxs:
        try: _add_cache_control(ms[i], ttl)
        except IndexError: continue

# %% ../nbs/00_core.ipynb
def mk_msgs(
    msgs,                   # List of messages (each: str, bytes, list, or dict w 'role' and 'content' fields)
    cache=False,            # Enable Anthropic caching
    cache_idxs=[-1],        # Cache breakpoint idxs
    ttl=None,               # Cache TTL: '5m' (default) or '1h'
):
    "Create a list of LiteLLM compatible messages."
    if not msgs: return []
    if not isinstance(msgs, list): msgs = [msgs]
    res,role = [],'user'
    msgs = L(msgs).map(lambda m: fmt2hist(m) if detls_tag in m else [m]).concat()
    for m in msgs:
        res.append(msg:=remove_cache_ckpts(mk_msg(m, role=role)))
        role = 'assistant' if msg['role'] in ('user','function', 'tool') else 'user'
    if cache: _apply_cache_idxs(res, cache_idxs, ttl)
    return res

# %% ../nbs/00_core.ipynb
def stream_with_complete(gen, postproc=noop):
    "Extend streaming response chunks with the complete response"
    chunks = []
    for chunk in gen:
        chunks.append(chunk)
        yield chunk
    postproc(chunks)
    return stream_chunk_builder(chunks)

# %% ../nbs/00_core.ipynb
def lite_mk_func(f):
    if isinstance(f, dict): return f
    return {'type':'function', 'function':get_schema(f, pname='parameters')}

# %% ../nbs/00_core.ipynb
@dataclass
class ToolResponse:
    content: list[str,str]

# %% ../nbs/00_core.ipynb
def _lite_call_func(tc,ns,raise_on_err=True):
    try: fargs = json.loads(tc.function.arguments)
    except Exception as e: raise ValueError(f"Failed to parse function arguments: {tc.function.arguments}") from e
    res = call_func(tc.function.name, fargs,ns=ns)
    if isinstance(res, ToolResponse): res = res.content
    else: res = str(res)
    return {"tool_call_id": tc.id, "role": "tool", "name": tc.function.name, "content": res}

# %% ../nbs/00_core.ipynb
def _has_search(m):
    i = get_model_info(m)
    return bool(i.get('search_context_cost_per_query') or i.get('supports_web_search'))

# %% ../nbs/00_core.ipynb
def cite_footnote(msg):
    if not (delta:=nested_idx(msg, 'choices', 0, 'delta')): return
    if citation:= nested_idx(delta, 'provider_specific_fields', 'citation'):
        title = citation['title'].replace('"', '\\"')
        delta.content = f'[*]({citation["url"]} "{title}") '
        
def cite_footnotes(stream_list):
    "Add markdown footnote citations to stream deltas"
    for msg in stream_list: cite_footnote(msg)

# %% ../nbs/00_core.ipynb
effort = AttrDict({o[0]:o for o in ('low','medium','high')})

# %% ../nbs/00_core.ipynb
def _mk_prefill(pf): return ModelResponseStream([StreamingChoices(delta=Delta(content=pf,role='assistant'))])

# %% ../nbs/00_core.ipynb
_final_prompt = "You have no more tool uses. Please summarize your findings. If you did not complete your goal please tell the user what further work needs to be done so they can choose how best to proceed."

# %% ../nbs/00_core.ipynb
class Chat:
    def __init__(
        self,
        model:str,                # LiteLLM compatible model name 
        sp='',                    # System prompt
        temp=0,                   # Temperature
        search=False,             # Search (l,m,h), if model supports it
        tools:list=None,          # Add tools
        hist:list=None,           # Chat history
        ns:Optional[dict]=None,   # Custom namespace for tool calling 
        cache=False,              # Anthropic prompt caching
        cache_idxs:list=[-1],     # Anthropic cache breakpoint idxs, use `0` for sys prompt if provided
        ttl=None,                 # Anthropic prompt caching ttl
    ):
        "LiteLLM chat client."
        self.model = model
        hist,tools = mk_msgs(hist,cache,cache_idxs,ttl),listify(tools)
        if ns is None and tools: ns = mk_ns(tools)
        elif ns is None: ns = globals()
        self.tool_schemas = [lite_mk_func(t) for t in tools] if tools else None
        store_attr()
    
    def _prep_msg(self, msg=None, prefill=None):
        "Prepare the messages list for the API call"
        sp = [{"role": "system", "content": self.sp}] if self.sp else []
        if sp:
            if 0 in self.cache_idxs: sp[0] = _add_cache_control(sp[0])
            cache_idxs = L(self.cache_idxs).filter().map(lambda o: o-1 if o>0 else o)
        else:
            cache_idxs = self.cache_idxs
        if msg: self.hist = mk_msgs(self.hist+[msg], self.cache, cache_idxs, self.ttl)
        pf = [{"role":"assistant","content":prefill}] if prefill else []
        return sp + self.hist + pf

    def _call(self, msg=None, prefill=None, temp=None, think=None, search=None, stream=False, max_steps=2, step=1, final_prompt=None, tool_choice=None, **kwargs):
        "Internal method that always yields responses"
        if step>max_steps: return
        if not get_model_info(self.model).get("supports_assistant_prefill"): prefill=None
        if _has_search(self.model) and (s:=ifnone(search,self.search)): kwargs['web_search_options'] = {"search_context_size": effort[s]}
        else: _=kwargs.pop('web_search_options',None)
        res = completion(model=self.model, messages=self._prep_msg(msg, prefill), stream=stream, 
                         tools=self.tool_schemas, reasoning_effort = effort.get(think), tool_choice=tool_choice,
                         # temperature is not supported when reasoning
                         temperature=None if think else ifnone(temp,self.temp),
                         **kwargs)
        if stream:
            if prefill: yield _mk_prefill(prefill)
            res = yield from stream_with_complete(res,postproc=cite_footnotes)
        m = res.choices[0].message
        if prefill: m.content = prefill + m.content
        self.hist.append(m)
        yield res

        if tcs := m.tool_calls:
            tool_results=[_lite_call_func(tc, ns=self.ns) for tc in tcs]
            self.hist+=tool_results
            for r in tool_results: yield r
            if step>=max_steps-1: prompt,tool_choice,search = final_prompt,'none',False
            else: prompt = None
            yield from self._call(
                prompt, prefill, temp, think, search, stream, max_steps, step+1,
                final_prompt, tool_choice, **kwargs)
    
    def __call__(self,
                 msg=None,          # Message str, or list of multiple message parts
                 prefill=None,      # Prefill AI response if model supports it
                 temp=None,         # Override temp set on chat initialization
                 think=None,        # Thinking (l,m,h)
                 search=None,       # Override search set on chat initialization (l,m,h)
                 stream=False,      # Stream results
                 max_steps=2, # Maximum number of tool calls
                 final_prompt=_final_prompt, # Final prompt when tool calls have ran out 
                 return_all=False,  # Returns all intermediate ModelResponses if not streaming and has tool calls
                 **kwargs):
        "Main call method - handles streaming vs non-streaming"
        result_gen = self._call(msg, prefill, temp, think, search, stream, max_steps, 1, final_prompt, **kwargs)     
        if stream: return result_gen              # streaming
        elif return_all: return list(result_gen)  # toolloop behavior
        else: return last(result_gen)             # normal chat behavior

# %% ../nbs/00_core.ipynb
@patch
def print_hist(self:Chat):
    "Print each message on a different line"
    for r in self.hist: print(r, end='\n\n')

# %% ../nbs/00_core.ipynb
def random_tool_id():
    "Generate a random tool ID with 'toolu_' prefix"
    random_part = ''.join(random.choices(string.ascii_letters + string.digits, k=25))
    return f'toolu_{random_part}'

# %% ../nbs/00_core.ipynb
def mk_tc(func, args, tcid=None, idx=1):
    if not tcid: tcid = random_tool_id()
    return {'index': idx, 'function': {'arguments': args, 'name': func}, 'id': tcid, 'type': 'function'}

# %% ../nbs/00_core.ipynb
def mk_tc_req(content, tcs):
    msg = Message(content=content, role='assistant', tool_calls=tcs, function_call=None)
    msg.tool_calls = [{**dict(tc), 'function': dict(tc['function'])} for tc in msg.tool_calls]
    return msg

# %% ../nbs/00_core.ipynb
def mk_tc_result(tc, result): return {'tool_call_id': tc['id'], 'role': 'tool', 'name': tc['function']['name'], 'content': result}

# %% ../nbs/00_core.ipynb
def mk_tc_results(tcq, results): return [mk_tc_result(a,b) for a,b in zip(tcq.tool_calls, results)]

# %% ../nbs/00_core.ipynb
async def _alite_call_func(tc, ns, raise_on_err=True):
    try: fargs = json.loads(tc.function.arguments)
    except Exception as e: raise ValueError(f"Failed to parse function arguments: {tc.function.arguments}") from e
    res = await call_func_async(tc.function.name, fargs, ns=ns)
    if isinstance(res, ToolResponse): res = res.content
    else: res = str(res)
    return {"tool_call_id": tc.id, "role": "tool", "name": tc.function.name, "content": res}

# %% ../nbs/00_core.ipynb
@asave_iter
async def astream_with_complete(self, agen, postproc=noop):
    chunks = []
    async for chunk in agen:
        chunks.append(chunk)
        postproc(chunk)
        yield chunk
    self.value = stream_chunk_builder(chunks)

# %% ../nbs/00_core.ipynb
class AsyncChat(Chat):
    async def _call(self, msg=None, prefill=None, temp=None, think=None, search=None, stream=False, max_steps=2, step=1, final_prompt=None, tool_choice=None, **kwargs):
        if step>max_steps+1: return
        if not get_model_info(self.model).get("supports_assistant_prefill"): prefill=None
        if _has_search(self.model) and (s:=ifnone(search,self.search)): kwargs['web_search_options'] = {"search_context_size": effort[s]}
        else: _=kwargs.pop('web_search_options',None)
        res = await acompletion(model=self.model, messages=self._prep_msg(msg, prefill), stream=stream,
                         tools=self.tool_schemas, reasoning_effort=effort.get(think), tool_choice=tool_choice,
                         # temperature is not supported when reasoning
                         temperature=None if think else ifnone(temp,self.temp), 
                         **kwargs)
        if stream:
            if prefill: yield _mk_prefill(prefill)
            res = astream_with_complete(res,postproc=cite_footnote)
            async for chunk in res: yield chunk
            res = res.value
        m=res.choices[0].message
        if prefill: m.content = prefill + m.content
        yield res
        self.hist.append(m)

        if tcs := m.tool_calls:
            tool_results = []
            for tc in tcs:
                result = await _alite_call_func(tc, ns=self.ns)
                tool_results.append(result)
                yield result
            self.hist+=tool_results
            if step>=max_steps-1: prompt,tool_choice,search = final_prompt,'none',False
            else: prompt = None
            async for result in self._call(
                prompt, prefill, temp, think, search, stream, max_steps, step+1,
                final_prompt, tool_choice=tool_choice, **kwargs):
                    yield result
    
    async def __call__(self,
                       msg=None,          # Message str, or list of multiple message parts
                       prefill=None,      # Prefill AI response if model supports it
                       temp=None,         # Override temp set on chat initialization
                       think=None,        # Thinking (l,m,h)
                       search=None,       # Override search set on chat initialization (l,m,h)
                       stream=False,      # Stream results
                       max_steps=2, # Maximum number of tool calls
                       final_prompt=_final_prompt, # Final prompt when tool calls have ran out 
                       return_all=False,  # Returns all intermediate ModelResponses if not streaming and has tool calls
                       **kwargs):
        result_gen = self._call(msg, prefill, temp, think, search, stream, max_steps, 1, final_prompt, **kwargs)
        if stream or return_all: return result_gen
        async for res in result_gen: pass
        return res # normal chat behavior only return last msg

# %% ../nbs/00_core.ipynb
def _trunc_str(s, mx=2000, replace="<TRUNCATED>"):
    "Truncate `s` to `mx` chars max, adding `replace` if truncated"
    s = str(s).strip()
    return s[:mx]+replace if len(s)>mx else s

# %% ../nbs/00_core.ipynb
def mk_tr_details(tr, tc, mx=2000):
    "Create <details> block for tool call as JSON"
    args = {k:_trunc_str(v, mx=mx) for k,v in json.loads(tc.function.arguments).items()}
    res = {'id':tr['tool_call_id'], 
           'call':{'function': tc.function.name, 'arguments': args},
           'result':_trunc_str(tr.get('content'), mx=mx),}
    return f"\n\n{detls_tag}\n\n```json\n{dumps(res, indent=2)}\n```\n\n</details>\n\n"

# %% ../nbs/00_core.ipynb
class AsyncStreamFormatter:
    def __init__(self, include_usage=False, mx=2000):
        self.outp,self.tcs,self.include_usage,self.think,self.mx = '',{},include_usage,False,mx
    
    def format_item(self, o):
        "Format a single item from the response stream."
        res = ''
        if isinstance(o, ModelResponseStream):
            d = o.choices[0].delta
            if nested_idx(d, 'reasoning_content'): 
                self.think = True
                res += '🧠'
            elif self.think:
                self.think = False
                res += '\n\n'
            if c:=d.content: res+=c
        elif isinstance(o, ModelResponse):
            if self.include_usage: res += f"\nUsage: {o.usage}"
            if c:=getattr(o.choices[0].message,'tool_calls',None):
                self.tcs = {tc.id:tc for tc in c}
        elif isinstance(o, dict) and 'tool_call_id' in o:
            res += mk_tr_details(o, self.tcs.pop(o['tool_call_id']), mx=self.mx)
        self.outp+=res
        return res
    
    async def format_stream(self, rs):
        "Format the response stream for markdown display."
        async for o in rs: yield self.format_item(o)

# %% ../nbs/00_core.ipynb
async def adisplay_stream(rs):
    "Use IPython.display to markdown display the response stream."
    try: from IPython.display import display, Markdown
    except ModuleNotFoundError: raise ModuleNotFoundError("This function requires ipython. Please run `pip install ipython` to use.")
    fmt = AsyncStreamFormatter()
    md = ''
    async for o in fmt.format_stream(rs): 
        md+=o
        display(Markdown(md),clear=True)
    return fmt

## Enrich doc
"""Fix, clean markdown headings and enrich it with figures description, ..."""

# AUTOGENERATED! DO NOT EDIT! File to edit: ../nbs/04_enrichr.ipynb.

# %% auto 0
__all__ = ['ANTHROPIC_API_KEY', 'GEMINI_API_KEY', 'cfg', 'src_dir', 'log_dir', 'lm', 'setup_enhanced_dir', 'get_hdgs',
           'get_hdgs_with_pages', 'format_hdgs', 'HeadingResult', 'FixHeadingHierarchy', 'fix_md',
           'group_corrections_by_page', 'apply_corrections_to_page', 'apply_all_corrections', 'fix_doc_hdgs',
           'has_images', 'MarkdownPage', 'ImgRef', 'ImageRelevance', 'describe_img', 'copy_page_to_enriched',
           'process_single_page', 'enrich_images', 'md_plus_evaluation']

# %% ../nbs/04_enrichr.ipynb 3
from pathlib import Path
import os
import re
import json
import shutil
import time
from functools import partial
from tqdm import tqdm
from dotenv import load_dotenv
from fastcore.all import *
import dspy
from pydantic import BaseModel
from typing import List
from litellm import completion
import base64
from rich import print
import logging

# %% ../nbs/04_enrichr.ipynb 4
load_dotenv()
ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')

# %% ../nbs/04_enrichr.ipynb 5
cfg = AttrDict({
    'enhanced_dir': 'enhanced',
    'enriched_dir': 'enriched',
    'lm': 'gemini/gemini-2.0-flash',
    'api_key': GEMINI_API_KEY,
    'max_tokens': 8192,
    'track_usage': False,
    'img_dir': 'img'
})

# %% ../nbs/04_enrichr.ipynb 6
src_dir = Path("../_data/md_library/49d2fba781b6a7c0d94577479636ee6f")

# %% ../nbs/04_enrichr.ipynb 7
log_dir = Path.home() / '.evaluatr'
log_dir.mkdir(exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(str(log_dir / 'enrichr.log')),
        logging.StreamHandler()
    ]
)

# %% ../nbs/04_enrichr.ipynb 10
def setup_enhanced_dir(
    src_dir, # Source directory path
    enhanced_dir_name=cfg.enhanced_dir # Name of enhanced subdirectory
    ):
    "Create enhanced directory and copy all markdown files to it"
    src_path = Path(src_dir)
    enhanced_path = src_path / enhanced_dir_name
    enhanced_path.mkdir(exist_ok=True)
    for f in src_path.ls(file_exts=".md"): shutil.copy(f, enhanced_path)
    return enhanced_path

# %% ../nbs/04_enrichr.ipynb 13
def get_hdgs(md_txt): return re.findall(r'^#+.*$', md_txt, re.MULTILINE)

# %% ../nbs/04_enrichr.ipynb 14
def get_hdgs_with_pages(
    pages: list[Path] # List of pages
    ):
    "Get headings and the page number they are on"
    headings = []
    for i, page in enumerate(pages, 1):  # page numbers start at 1
        page_headings = get_hdgs(page.read_text())
        for o in page_headings: headings.append({'heading': o, 'page': i})
    return headings

# %% ../nbs/04_enrichr.ipynb 17
def format_hdgs(
    hdgs: list[dict] # List of headings with page numbers
    ):
    "Format headings with page numbers"
    formatted = []
    page_positions = {}
    
    for item in hdgs:
        page = item['page']
        page_positions[page] = page_positions.get(page, 0) + 1
        formatted.append(f"{item['heading']} (Page {page}, Position {page_positions[page]})")
    
    return "\n".join(formatted)

# %% ../nbs/04_enrichr.ipynb 19
lm = dspy.LM(cfg.lm, api_key=cfg.api_key)
dspy.configure(lm=lm)
dspy.settings.configure(track_usage=cfg.track_usage)

# %% ../nbs/04_enrichr.ipynb 20
class HeadingResult(BaseModel):
    old: str
    page: int
    position: int
    new: str
    changed: bool  # True if correction was made

# %% ../nbs/04_enrichr.ipynb 21
class FixHeadingHierarchy(dspy.Signature):
    """Fix markdown heading hierarchy by analyzing the document's numbering patterns:
    - Detect numbering scheme (1.2.3, I.A.1, A.1.a, etc.)
    - Apply hierarchy levels based on nesting depth: # for top level, ## for second level, ### for third level
    - When a section number is lower than a previously seen number at the same level (e.g., seeing '2.' after '3.1'), it's likely a subsection or list item, not a main section
    - Unnumbered headings: keep as-is if at document boundaries, treat as subsections if within numbered sections
    - Return ALL headings with their corrected form
    """
    
    headings_with_pages: str = dspy.InputField(desc="List of headings with page numbers")
    results: List[HeadingResult] = dspy.OutputField(desc="All headings with corrections and change status")

# %% ../nbs/04_enrichr.ipynb 22
def fix_md(
    hdgs: list[dict], # List of headings with page numbers
    track_usage: bool=cfg.track_usage,
    ):
    "Fix markdown headings"
    lm = dspy.LM(cfg.lm, api_key=cfg.api_key, max_tokens=cfg.max_tokens)
    dspy.configure(lm=lm)
    dspy.settings.configure(track_usage=track_usage)

    inp = format_hdgs(hdgs)
    fix_hdgs = dspy.ChainOfThought(FixHeadingHierarchy)
    result = fix_hdgs(headings_with_pages=inp)
    return result

# %% ../nbs/04_enrichr.ipynb 24
def group_corrections_by_page(
    results: list[HeadingResult], # List of headings with corrections and change status
    ):
    "Group HeadingResult corrections by page number into dict with page nums as keys"
    page_groups = {}
    for result in results:
        page = result.page
        if page not in page_groups:
            page_groups[page] = []
        page_groups[page].append(result)
    return page_groups

# %% ../nbs/04_enrichr.ipynb 26
def apply_corrections_to_page(
    page_nb, # Page number
    corrections, # List of corrections
    enhanced_path, # Path to enhanced directory
    ):
    "Apply corrections to a page in the enhanced directory"
    page_file = enhanced_path / f"page_{page_nb}.md"
    lines = page_file.read_text().splitlines()
    corrections_copy = corrections.copy()
    
    for i, line in enumerate(lines):
        for correction in corrections_copy:
            if line.strip() == correction.old.strip():
                lines[i] = f"{correction.new} .... page {page_nb}"
                corrections_copy.remove(correction)
                break
            
    page_file.write_text('\n'.join(lines))

# %% ../nbs/04_enrichr.ipynb 28
def apply_all_corrections(
    results, # List of headings with corrections and change status
    enhanced_path, # Path to enhanced directory
    ):
    "Apply all corrections to the pages in enhanced directory"
    grouped = group_corrections_by_page(results)
    for page_nb, corrections in grouped.items(): 
        apply_corrections_to_page(page_nb, corrections, enhanced_path)

# %% ../nbs/04_enrichr.ipynb 30
def fix_doc_hdgs(
    src_dir, # Path to the folder containing the document
    force=False, # Whether to overwrite the existing enhanced directory
    ):
    "Process the document directory"
    src_path = Path(src_dir)
    enhanced_path = src_path / cfg.enhanced_dir
    
    if enhanced_path.exists() and not force:
        print(f"Enhanced directory '{cfg.enhanced_dir}' already exists. Use force=True to overwrite.")
        return
    if enhanced_path.exists() and force: 
        shutil.rmtree(enhanced_path)
    
    enhanced_path = setup_enhanced_dir(src_dir)
    pages = enhanced_path.ls(file_exts=".md").sorted(key=lambda p: int(p.stem.split('_')[1]))
    result = fix_md(get_hdgs_with_pages(pages))
    apply_all_corrections(result.results, enhanced_path)

# %% ../nbs/04_enrichr.ipynb 34
def has_images(page_path):
    content = Path(page_path).read_text()
    return bool(re.search(r'!\[[^\]]*\]\([^)]+\)', content))

# %% ../nbs/04_enrichr.ipynb 37
class MarkdownPage: 
    "A class to represent a markdown page"
    def __init__(self, path): self.path = Path(path)

# %% ../nbs/04_enrichr.ipynb 38
class ImgRef(AttrDict):
    "A class to represent a image reference"
    def __repr__(self):
        clean_context = self.context.replace('\n', ' ')[:50] + "..."
        fields = [f"filename='{self.filename}'", f"context='{clean_context}'"]
        if hasattr(self, 'is_relevant'): fields.append(f"is_relevant={self.is_relevant}")
        if hasattr(self, 'reason'): fields.append(f"reason={self.reason}")
        # ... add other fields if present
        return f"ImgRef({', '.join(fields)})"


# %% ../nbs/04_enrichr.ipynb 39
@patch
def find_img_refs(
    self:MarkdownPage, # Markdown page of interest
    context_lines: int = 3, # Number of lines of context to include around the image
    ):
    "Find all image references in the markdown page and include the context around the image"
    content = self.path.read_text()
    lines = content.splitlines()
    results = []
    
    for i, line in enumerate(lines):
        if re.search(r'!\[[^\]]*\]\(([^)]+)\)', line):
            # Extract context around this line
            start = max(0, i - context_lines)
            end = min(len(lines), i + context_lines + 1)
            context = '\n'.join(lines[start:end])
            
            # Extract image filename
            match = re.search(r'!\[[^\]]*\]\(([^)]+)\)', line)
            results.append(ImgRef({
                "filename": match.group(1),
                "context": context
            }))
    
    return results

# %% ../nbs/04_enrichr.ipynb 42
class ImageRelevance(dspy.Signature):
    """Determine if an image contains substantive content for document understanding.
    
    RELEVANT: Charts, graphs, diagrams, figures, tables, screenshots, flowcharts
    IRRELEVANT: Logos, cover images, decorative elements, headers, footers
    """
    img_filename: str = dspy.InputField()
    surrounding_context: str = dspy.InputField(desc="Text context around the image")
    is_relevant: bool = dspy.OutputField(desc="True only for substantive content like data visualizations")
    reason: str = dspy.OutputField(desc="Brief explanation of decision")


# %% ../nbs/04_enrichr.ipynb 43
@patch
def classify_imgs(
    self:MarkdownPage, # Markdown page of interest
    img_refs: list[ImgRef], # List of image references
    ):
    "Classify images in the markdown page"
    classifier = dspy.ChainOfThought(ImageRelevance)
    for img_ref in img_refs:
        result = classifier(
            img_filename=img_ref.filename,
            surrounding_context=img_ref.context,
            page_nb=1  # We could make this dynamic if needed
        )
        img_ref.is_relevant = result.is_relevant
        img_ref.reason = result.reason
    return img_refs

# %% ../nbs/04_enrichr.ipynb 46
def describe_img(
    img_path: Path, # Path to the image
    context: str, # Context of the image
    api_key: str = cfg.api_key, # API key for the LLM model
    model: str = cfg.lm, # Model to use
    ):
    "Describe an image using an LLM"
    with open(img_path, "rb") as image_file:
        base64_image = base64.b64encode(image_file.read()).decode('utf-8')
    
    # Auto-detect image format
    img_format = img_path.suffix.lower().replace('.', '')
    if img_format == 'jpg': img_format = 'jpeg'
    
    prompt = f"""Provide a concise paragraph description of this image for evaluation report analysis. Include: type of content, main topic, key data/statistics, trends, and takeaways. Write as flowing text, not numbered points. Context: {context}"""
    response = completion(
        model=model,
        messages=[{
            "role": "user", 
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": f"data:image/{img_format};base64,{base64_image}"}}
            ]
        }],
        api_key=api_key
    )
    return response.choices[0].message.content

# %% ../nbs/04_enrichr.ipynb 47
@patch
def describe_imgs(
    self:MarkdownPage, # Markdown page of interest
    img_refs: list[ImgRef], # List of image references
    img_dir: str # Image directory
    ):
    "Describe images in the markdown page"
    for img_ref in img_refs:
        if img_ref.is_relevant:
            img_path = Path(img_dir) / img_ref.filename
            description = describe_img(img_path, img_ref.context, GEMINI_API_KEY)
            img_ref.description = description
    return img_refs

# %% ../nbs/04_enrichr.ipynb 50
@patch
def replace_imgs_with_desc(
    self:MarkdownPage, # Markdown page of interest
    img_refs, # List of image references
    enriched_dir: str = cfg.enriched_dir, # Enriched directory
    ):
    "Replace images with their descriptions in the markdown page"
    enriched_path = self.path.parent.parent / enriched_dir
    enriched_path.mkdir(exist_ok=True)
    
    content = self.path.read_text()
    for img_ref in img_refs:
        if img_ref.is_relevant and hasattr(img_ref, 'description'):
            pattern = f'!\\[[^\\]]*\\]\\({re.escape(img_ref.filename)}\\)'
            content = re.sub(pattern, img_ref.description, content)
    
    enriched_file = enriched_path / self.path.name
    enriched_file.write_text(content)
    return enriched_file

# %% ../nbs/04_enrichr.ipynb 51
def copy_page_to_enriched(
    page, # Page to copy
    enriched_dir: str = cfg.enriched_dir, # Enriched directory
    ):
    "Copy a page to the enriched directory"
    enriched_path = page.parent.parent / enriched_dir
    enriched_path.mkdir(exist_ok=True)
    return shutil.copy(page, enriched_path)

# %% ../nbs/04_enrichr.ipynb 52
def process_single_page(
    page, # Page to process
    img_dir, # Image directory
    enriched_dir: str = cfg.enriched_dir, # Enriched directory
    ):
    "Process a single page"
    md_page = MarkdownPage(page)
    # Pipeline: find → classify → describe → replace
    img_refs = md_page.find_img_refs()
    
    if not img_refs: return copy_page_to_enriched(page, enriched_dir)
    
    classified_refs = md_page.classify_imgs(img_refs)
    time.sleep(0.5)
    described_refs = md_page.describe_imgs(classified_refs, img_dir)
    time.sleep(0.5)
    return md_page.replace_imgs_with_desc(described_refs)

# %% ../nbs/04_enrichr.ipynb 53
def enrich_images(
    pages_dir, # Pages directory
    img_dir, # Image directory
    n_workers=2, # Number of workers
    ):
    "Enrich images in the pages directory"
    pages = Path(pages_dir).ls(file_exts=".md")
    
    pages_with_imgs = []
    for page in pages:
        if has_images(page):
            pages_with_imgs.append(page)
        else:
            copy_page_to_enriched(page)
    
    if pages_with_imgs:
        process_fn = partial(process_single_page, img_dir=img_dir)
        parallel(process_fn, pages_with_imgs, n_workers=n_workers, threadpool=True, progress=True)
        
    print(f"✓ Processed {len(pages)} pages ({len(pages_with_imgs)} with images)")

# %% ../nbs/04_enrichr.ipynb 56
@call_parse
def md_plus_evaluation(
    eval_id: str,  # Evaluation ID to process
    md_dir: str = "../data/md_library",  # Directory containing markdown folders
    overwrite: bool = False  # Overwrite if enhanced/enriched already exists
):
    "Fix markdown headings and enrich images for an evaluation report"
    # Find the evaluation's markdown directory
    eval_path = Path(md_dir) / eval_id
    
    if not eval_path.exists():
        logging.error(f"Markdown directory not found: {eval_path}")
        return
    
    # Find the actual report folder (first subdirectory)
    report_dirs = [d for d in eval_path.iterdir() if d.is_dir() 
                   and d.name != cfg.enhanced_dir and d.name != cfg.enriched_dir]
    
    if not report_dirs:
        logging.error(f"No report directory found in {eval_path}")
        return
    
    doc_path = report_dirs[0]
    
    # Check if already processed
    enhanced_path = doc_path / cfg.enhanced_dir
    enriched_path = doc_path.parent / cfg.enriched_dir
    
    if (enhanced_path.exists() or enriched_path.exists()) and not overwrite:
        logging.info(f"Output already exists for {eval_id}. Use --overwrite to reprocess.")
        return
    
    # Clean up if overwriting
    if overwrite:
        if enhanced_path.exists():
            shutil.rmtree(enhanced_path)
        if enriched_path.exists():
            shutil.rmtree(enriched_path)
    
    logging.info(f"Processing {eval_id}: fixing headings...")
    fix_doc_hdgs(doc_path, force=overwrite)
    
    logging.info(f"Processing {eval_id}: enriching images...")
    pages_dir = doc_path / cfg.enhanced_dir
    img_dir = doc_path / cfg.img_dir
    enrich_images(pages_dir, img_dir, n_workers=1)
    
    logging.info(f"Completed processing {eval_id}")

In [ ]:
mappr = nb_to_md("_06_mappr.ipynb")
mappr[:200]

'# Mappr\n\n> Scale up evaluation report mapping against evaluation frameworks using agentic workflows\n\n\n---\n\n::: {.callout-warning}\nThis notebook is a work in progress.\n:::\n\n---\n\nManually mapping evalua'

In [ ]:
#| export
def get_hdgs(
    md:str # Markdown file string
    ):
    "Return the markdown headings"
    # Sanitize removing '#' in python snippet if any
    md = sub(r'```[\s\S]*?```', '', md)
    return findall(r'^#{1,6} .+$', md, MULTILINE)



In [ ]:
hdgs = get_hdgs(md)
hdgs

['# Deep Residual Learning for Image Recognition ',
 '#### Abstract',
 '## 1. Introduction',
 '## 2. Related Work',
 '## 3. Deep Residual Learning',
 '### 3.1. Residual Learning',
 '### 3.2. Identity Mapping by Shortcuts',
 '### 3.3. Network Architectures',
 '### 3.4. Implementation',
 '## 4. Experiments',
 '### 4.1. ImageNet Classification',
 '### 4.2. CIFAR-10 and Analysis',
 '### 4.3. Object Detection on PASCAL and MS COCO',
 '## References',
 '## A. Object Detection Baselines',
 '## PASCAL VOC',
 '## MS COCO',
 '## B. Object Detection Improvements',
 '## MS COCO',
 '## PASCAL VOC',
 '## ImageNet Detection',
 '## C. ImageNet Localization']

What if I want to ask an LLM to fix those markdown headings:

For instance:
```
['# Deep Residual Learning for Image Recognition ',
 '#### Abstract',
 '## 1. Introduction',
 '## 2. Related Work',
 '## 3. Deep Residual Learning',
 '### 3.1. Residual Learning',
 '### 3.2. Identity Mapping by Shortcuts',
 '### 3.3. Network Architectures',
 '### 3.4. Implementation',
 '## 4. Experiments',
 '### 4.1. ImageNet Classification',
 '### 4.2. CIFAR-10 and Analysis',
 '### 4.3. Object Detection on PASCAL and MS COCO',
 '## References',
 '## A. Object Detection Baselines',
 '## PASCAL VOC',
 '## MS COCO',
 '## B. Object Detection Improvements',
 '## MS COCO',
 '## PASCAL VOC',
 '## ImageNet Detection',
 '## C. ImageNet Localization']
```

We want to fix markdown heading hierarchy by analyzing the document's numbering patterns:
- Detect numbering scheme (1.2.3, I.A.1, A.1.a, etc.) if any
- Apply hierarchy levels based on nesting depth: # for top level, ## for second level, ### for third level
- When a section number is lower than a previously seen number at the same level (e.g., seeing '2.' after '3.1'), it's likely a subsection or list item, not a main section
- Unnumbered headings: keep as-is if at document boundaries, treat as subsections if within numbered sections
- Return ALL headings with their corrected form

There is also one rule that can be checked:
- subsection headings can only increase by one '#', not jump such as passing from "# Deep ..." to "#### Abstract"
- however we can definitely jump from '### 4.3. Object Detection on PASCAL and MS COCO' to "## References" 

Can you help me designing a clear an unambiguous prompt.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Great question! Let me help you design a clear prompt for this task.

First, a few clarifying questions:

1. **Output format**: Do you want the LLM to return a list of corrected headings in the same order, or would you prefer a structured format (like JSON) that shows both the original and corrected version for each heading?

2. **Edge cases**: In your example, "#### Abstract" should probably become "## Abstract" (since it follows the title). Should unnumbered headings after numbered sections (like "## PASCAL VOC" after "## A. Object Detection Baselines") become subsections (###)?

3. **Validation**: Should "## References" stay as ## even though it's unnumbered, because it's a standard section that appears after main content?

These clarifications will help me craft a more precise prompt. What are your thoughts?

1. For the output format, ideally I'd like to have a dictionary with `{'original heading': 'fixed heading', ...}` (though it might break if similar headings in the document - but should be a rare limit case)
2. yes, I agree on the "Abstract" example here, it should be definitely become '##'. Yes, your are right "## PASCAL VOC" should become "### PASCAL VOC"
3. yes, "References" should remain "## References" even if unumbered

This might not work well on all edge cases but as long as it's already better than the original version and fixed "obvious" cases it's already an added value.


##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Perfect! Here's a draft prompt for the LLM:

```python
prompt = """Fix the markdown heading hierarchy in this list of headings.

Rules:
1. Detect numbering schemes (1.2.3, A.B.C, etc.) and apply hierarchy:
   - # for top-level (title or section 1, 2, 3...)
   - ## for second-level (sections 1.1, 1.2, A, B...)
   - ### for third-level (sections 1.1.1, A.1, etc.)

2. Headings can only increase by one level (# → ##, not # → ####)

3. Unnumbered headings:
   - Keep standard sections like "Abstract", "References" as ## 
   - After numbered sections, unnumbered headings become subsections (add one #)

4. Return a Python dictionary: {'original': 'fixed', ...}

Headings:
{headings_list}
"""
```

Would you like to test this prompt with your example, or would you like me to refine any part of it first?

# mappr
"""Scale up evaluation report mapping against evaluation frameworks using agentic workflows"""

# AUTOGENERATED! DO NOT EDIT! File to edit: ../nbs/06_mappr.ipynb.

# %% auto 0
__all__ = ['GEMINI_API_KEY', 'cfg', 'traces_dir', 'db_path', 'db', 'sections_cache', 'theme_cache', 'select_section_sp',
           'tagging_sp', 'tagging_sp_no_citation', 'SectionsCache', 'ThemeTaggingCache', 'load_report',
           'find_section_path', 'get_content_tool', 'flatten_sections', 'format_toc_for_llm', 'format_enabler_theme',
           'format_crosscutting_theme', 'format_gcm_theme', 'format_srf_output', 'CoreSectionsOutput',
           'EvidenceLocation', 'ThemeTaggingOutput', 'parse_response', 'identify_core_sections', 'extract_core_content',
           'TagResult', 'tag_theme', 'Stage', 'TraceContext', 'setup_logger', 'setup_trace_logging',
           'log_analysis_event', 'get_from_cache', 'store_in_cache', 'limit', 'TaggingResult', 'PipelineResults',
           'PipelineOrchestrator', 'get_filtered_srf_output_ids', 'find_enriched_path', 'parse_force_refresh',
           'run_selected_stages', 'tag_evaluation']

# %% ../nbs/06_mappr.ipynb 5
from pathlib import Path
from functools import reduce
from toolslm.md_hier import *
from rich import print
import json
from fastcore.all import *
from enum import Enum
import logging
import uuid
from datetime import datetime
from typing import List, Callable
import dspy
import time
from collections import defaultdict
import copy
from copy import deepcopy
from dataclasses import dataclass
from typing import List
from pydantic import BaseModel, Field
import asyncio
from asyncio import Semaphore, gather, sleep

from .frameworks import (EvalData, 
                                 IOMEvalData, 
                                 FrameworkInfo, 
                                 Framework,
                                 FrameworkCat,
                                 find_srf_output_by_id)

from fastlite import Database
from apswutils.db import NotFoundError

from lisette import mk_msg, AsyncChat
from lisette.core import acompletion

# %% ../nbs/06_mappr.ipynb 6
from dotenv import load_dotenv
import os

load_dotenv()
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')

# %% ../nbs/06_mappr.ipynb 7
cfg = AttrDict({
    'lm': 'gemini/gemini-2.0-flash',
    'api_key': GEMINI_API_KEY,
    'max_tokens': 8192,
    'track_usage': False,
    'call_delay': 0.1, # in seconds
    'semaphore': 2,
    'dirs': AttrDict({
        'data': '.evaluatr',
        'trace': 'traces'
    }),
    'verbosity': 1,
    'cache': AttrDict({
        'db_name':  'pipeline_cache.db'
    }),
})

# %% ../nbs/06_mappr.ipynb 8
traces_dir = Path.home() / cfg.dirs.data / cfg.dirs.trace
traces_dir.mkdir(parents=True, exist_ok=True)

# %% ../nbs/06_mappr.ipynb 10
db_path = traces_dir / cfg.cache.db_name
db = Database(db_path)

# %% ../nbs/06_mappr.ipynb 11
@dataclass
class SectionsCache:
    report_id: str
    sections_selected: str  # JSON list
    reasoning: str
    timestamp: str = None

# %% ../nbs/06_mappr.ipynb 12
@dataclass
class ThemeTaggingCache:
    report_id: str
    stage: str
    framework: str
    framework_category: str
    framework_theme_id: str
    is_core: bool
    reasoning: str
    confidence: str
    timestamp: str = None

# %% ../nbs/06_mappr.ipynb 13
sections_cache = db.create(SectionsCache, pk='report_id', ignore=True)
theme_cache = db.create(
    ThemeTaggingCache, 
    pk=['report_id', 'stage', 'framework', 'framework_category', 'framework_theme_id'],
    ignore=True
)

# %% ../nbs/06_mappr.ipynb 15
def load_report(
    doc_path:str # Path to the evaluation report
    ):
    "Read evaluation report from enriched markdown pages"
    doc = Path(doc_path)
    pages = doc.ls(file_exts=".md").sorted(key=lambda p: int(p.stem.split('_')[1]))
    report = '\n\n---\n\n'.join(page.read_text() for page in pages)
    return report

# %% ../nbs/06_mappr.ipynb 20
def find_section_path(
    hdgs: dict, # The nested dictionary structure
    target_section: str # The section name to find
) -> list: # The nested key path for the given section name
    "Find the nested key path for a given section name."
    def search_recursive(current_dict, path=[]):
        for key, value in current_dict.items():
            current_path = path + [key]
            if key == target_section:
                return current_path
            if isinstance(value, dict):
                result = search_recursive(value, current_path)
                if result:
                    return result
        return None
    
    return search_recursive(hdgs)

# %% ../nbs/06_mappr.ipynb 24
def get_content_tool(
    hdgs: dict, # The nested dictionary structure
    keys_list: list, # The list of keys to navigate through
    ) -> str: # The content of the section
    "Navigate through nested levels using the exact key strings."
    return reduce(lambda current, key: current[key], keys_list, hdgs).text

# %% ../nbs/06_mappr.ipynb 26
def flatten_sections(
    hdgs: dict, # The nested dictionary-like structure of the report also allowing to pull content from 
    path: list = [] # The current path in the nested structure
    ) -> list: # The flat list of (key, full_path) tuples
    "Extract flat list of (key, full_path) tuples from nested hdgs"
    sections = []
    for key, value in hdgs.items():
        current_path = path + [key]
        sections.append((key, current_path))
        if isinstance(value, dict):
            sections.extend(flatten_sections(value, current_path))
    return sections

# %% ../nbs/06_mappr.ipynb 28
def format_toc_for_llm(hdgs: dict) -> str:
    """Format ToC as readable text with page numbers"""
    sections = flatten_sections(hdgs)
    lines = [f"- {key}" for key, path in sections]
    return '\n'.join(lines)

# %% ../nbs/06_mappr.ipynb 32
def format_enabler_theme(
    theme: EvalData # The theme object
    ) -> str: # The formatted theme string
    "Format SRF enabler into structured text for LM processing."
    parts = [
        f'## Enabler {theme.id}: {theme.title}',
        '### Description', 
        theme.description
    ]
    return '\n'.join(parts)

# %% ../nbs/06_mappr.ipynb 35
def format_crosscutting_theme(
    theme: EvalData # The theme object
    ) -> str: # The formatted theme string
    "Format SRF cross-cutting into structured text for LM processing."
    parts = [
        f'## Cross-cutting {theme.id}: {theme.title}',
        '### Description', 
        theme.description
    ]
    return '\n'.join(parts)

# %% ../nbs/06_mappr.ipynb 38
def format_gcm_theme(
    theme: dict # The GCM theme object from gcm_small
    ) -> str: # The formatted theme string
    "Format GCM objective into structured text for LM processing."
    parts = [
        f'## GCM Objective {theme["id"]}: {theme["title"]}',
        '### Core Theme', 
        theme["core_theme"]
    ]
    
    if theme.get("key_principles"):
        parts.extend(['### Key Principles', ', '.join(theme["key_principles"])])
    
    if theme.get("target_groups"):
        parts.extend(['### Target Groups', ', '.join(theme["target_groups"])])
        
    if theme.get("main_activities"):
        parts.extend(['### Main Activities', ', '.join(theme["main_activities"])])
    
    return '\n'.join(parts)

# %% ../nbs/06_mappr.ipynb 42
def format_srf_output(output_context: dict) -> str:
    "Format SRF output with full hierarchical context for LM processing."
    parts = [
        f'## SRF Output {output_context["output"]["id"]}: {output_context["output"]["title"]}',
        '### Strategic Context',
        f'**Objective {output_context["objective"]["id"]}**: {output_context["objective"]["title"]}',
        f'**Long    -term Outcome {output_context["long_outcome"]["id"]}**: {output_context["long_outcome"]["title"]}',
        f'**Short-term Outcome {output_context["short_outcome"]["id"]}**: {output_context["short_outcome"]["title"]}'
    ]
    
    return '\n'.join(parts)

# %% ../nbs/06_mappr.ipynb 46
class CoreSectionsOutput(BaseModel):
    "Identify the core sections of the report"
    section_names: list[str]
    reasoning: str

# %% ../nbs/06_mappr.ipynb 47
class EvidenceLocation(BaseModel):
    "Identify the location of the evidence in the report"
    section: str
    citation: str

# %% ../nbs/06_mappr.ipynb 48
class ThemeTaggingOutput(BaseModel):
    "Tag the theme in the report"
    is_core: bool
    reasoning: str
    evidence_locations: list[EvidenceLocation]
    confidence: str  # low/medium/high

# %% ../nbs/06_mappr.ipynb 50
select_section_sp = """### ROLE AND OBJECTIVE
You are an expert evaluation report analyst. Your task is to identify sections that would help determine if specific themes are CORE to this report for synthesis and retrieval purposes.

### CONTEXT
You will receive a table of contents (ToC) with section headings from an evaluation report. Select sections where report authors signal what matters most - these will be used to tag themes for future synthesis work.

### SECTIONS TO IDENTIFY
Look for sections that reveal core themes (in any language):
1. Executive Summary / Overview / Résumé exécutif / Resumen ejecutivo
2. Introduction / Objectives / Purpose / Questions d'évaluation / Preguntas de evaluación
3. Main Findings / Results / Résultats / Resultados / Constatations
4. Conclusions / Conclusiones
5. Recommendations / Recommandations / Recomendaciones

### SELECTION CRITERIA
- Match flexibly by meaning, not exact wording
- Prioritize where authors explicitly state what's important
- Aim for ~8-10 pages total (use page numbers in ToC as guide)
- Avoid methodology, background, annexes unless unusually central
- Not all report types have all sections - select what exists

### OUTPUT FORMAT
JSON with section_names (list) and reasoning (string).

**CRITICAL**: section_names must contain EXACT strings from the ToC provided.
Copy the complete line including section numbers and page references.
Example: If ToC shows "4.1. Relevance of programme activities .... page 34"
Return exactly: "4.1. Relevance of programme activities .... page 34"
"""

# %% ../nbs/06_mappr.ipynb 51
tagging_sp = """### ROLE AND OBJECTIVE
You are an evaluation synthesis specialist. Your task is to determine if this report should be tagged with a specific theme for future retrieval in synthesis work.

### CONTEXT
You will receive:
- Key sections from an evaluation report
- A specific theme to evaluate

### CRITICAL DISTINCTION
You are NOT evaluating whether the theme is mentioned or relevant.
You ARE evaluating whether the theme is CENTRAL to what this report is fundamentally about.

### TAGGING DECISION CRITERIA
Tag as CORE only if the theme meets BOTH conditions:

**1. Centrality Test**: The theme is a PRIMARY focus of the report:
- The theme appears in the report's main objectives/evaluation questions
- Multiple major sections dedicate substantial analysis to this theme
- Key findings and conclusions center on this theme
- Major recommendations address this theme

**2. Synthesis Value Test**: Ask yourself:
"If I were synthesizing evaluation findings specifically on [Theme X], would EXCLUDING this report create a significant gap in my synthesis?"

### DECISION RULE
- Tag as CORE: The report would be among the TOP sources for a synthesis on this theme
- Tag as NOT CORE: The report mentions the theme but isn't fundamentally about it

**When uncertain → Tag as NOT CORE**

Aim for precision: Only 2-4 themes per report should be CORE.

### OUTPUT FORMAT
JSON with:
- is_core: boolean
- reasoning: explain centrality (or lack thereof) with specific evidence
- evidence_locations: list of {"section": "...", "citation": "..."}
- confidence: low/medium/high
"""

# %% ../nbs/06_mappr.ipynb 52
tagging_sp_no_citation = """### ROLE AND OBJECTIVE
You are an evaluation synthesis specialist. Your task is to determine if this report should be tagged with a 
specific theme for future retrieval in synthesis work.

### CONTEXT
You will receive:
- Key sections from an evaluation report
- A specific theme to evaluate

### CRITICAL DISTINCTION
You are NOT evaluating whether the theme is mentioned or relevant.
You ARE evaluating whether the theme is CENTRAL to what this report is fundamentally about.

### TAGGING DECISION CRITERIA
Tag as CORE only if the theme meets BOTH conditions:

**1. Centrality Test**: The theme is a PRIMARY focus of the report:
- The theme appears in the report's main objectives/evaluation questions
- Multiple major sections dedicate substantial analysis to this theme
- Key findings and conclusions center on this theme
- Major recommendations address this theme

**2. Synthesis Value Test**: Ask yourself:
"If I were synthesizing evaluation findings specifically on [Theme X], would EXCLUDING this report create a 
significant gap in my synthesis?"

### DECISION RULE
- Tag as CORE: The report would be among the TOP sources for a synthesis on this theme
- Tag as NOT CORE: The report mentions the theme but isn't fundamentally about it

**When uncertain → Tag as NOT CORE**

Aim for precision: Only 2-4 themes per report should be CORE.

### OUTPUT FORMAT
JSON with:
- is_core: boolean
- reasoning: explain centrality (or lack thereof) with specific evidence (max 150 words)
- confidence: low/medium/high"""

# %% ../nbs/06_mappr.ipynb 54
def parse_response(result):
    "Extract JSON from Lisette response"
    return json.loads(result.choices[0].message.content)

# %% ../nbs/06_mappr.ipynb 56
async def identify_core_sections(
    hdgs: dict, # The nested dictionary-like structure of the report also allowing to pull content from 
    system_prompt: str, # The system prompt for the core sections identification
    model: str = 'gemini/gemini-2.0-flash' # The model to use
) -> dict: # The JSON response from the model
    chat = AsyncChat(model=model, sp=system_prompt, temp=0)
    toc_text = format_toc_for_llm(hdgs)
    result = await chat(
        f"Here is the table of contents:\n\n{toc_text}",
        response_format=CoreSectionsOutput
    )
    return parse_response(result)

# %% ../nbs/06_mappr.ipynb 58
def extract_core_content(
    core_section_names: list[str],  # Section names from Step 1
    hdgs: dict  # Nested heading structure
) -> str:  # Combined content with section headers
    "Extract and combine content from core sections"
    if not core_section_names:
        logging.warning("No core sections provided")
        return ""
    
    sections_lookup = {key: path for key, path in flatten_sections(hdgs)}
    content_parts = []
    
    for section_name in core_section_names:
        if section_name not in sections_lookup:
            logging.warning(f"Section not found: {section_name}")
            continue
        path = sections_lookup[section_name]
        content = get_content_tool(hdgs, path)
        content_parts.append(content)
    
    return "\n\n---\n\n".join(content_parts)

# %% ../nbs/06_mappr.ipynb 62
class TagResult(BaseModel):
    "The result of the theme tagging"
    is_core: bool
    reasoning: str
    confidence: str

# %% ../nbs/06_mappr.ipynb 63
async def tag_theme(
    doc_content: str, # The content of the document to analyze
    theme: str, # The theme to tag
    system_prompt: str, # The system prompt for the theme tagging
    response_format: type = TagResult, # The response format for the theme tagging
    model: str = "claude-sonnet-4-5", # The model to use
    cache_system: bool = True,   # Always cache system+doc (default)
    cache_theme: bool = False     # Only cache theme for resume
):
    "Tag a single theme against the document"
    system_blocks = [
        {"type": "text", "text": system_prompt},
        {"type": "text", "text": f"\n\n## Document to Analyze\n\n{doc_content}"}
    ]
    
    if cache_system: system_blocks[-1]["cache_control"] = {"type": "ephemeral"}
    
    messages = [mk_msg(theme, cache=cache_theme)]
    
    response = await acompletion(
        model=model,
        messages=messages,
        system=system_blocks,
        response_format=response_format
    )
    return response

# %% ../nbs/06_mappr.ipynb 66
class Stage(Enum):
    "Pipeline stage number"
    STAGE1 = "stage1"
    STAGE2 = "stage2"
    STAGE3 = "stage3"
    def __str__(self): return self.value

# %% ../nbs/06_mappr.ipynb 68
class TraceContext(AttrDict):
    "Context for tracing the mapping process"
    def __init__(self, 
                 report_id:str,  # Report identifier
                 stage:Stage,  # Pipeline stage number
                 framework:FrameworkInfo,  # Framework info (name, category, theme_id)
                 ): 
        store_attr()

    def __repr__(self):
        return f"TraceContext(report_id={self.report_id}, stage={self.stage}, framework={self.framework})"

# %% ../nbs/06_mappr.ipynb 70
def setup_logger(
    name: str, # The name of the logger
    handler: logging.Handler, # The handler to use
    level: int = logging.INFO, # The level of the logger
    **kwargs: dict # Additional keyword arguments
    ):
    "Helper function to setup a logger with common configuration"
    logger = logging.getLogger(name)
    logger.handlers.clear()
    logger.addHandler(handler)
    logger.setLevel(level)
    for k,v in kwargs.items(): setattr(logger, k, v)
    return logger

# %% ../nbs/06_mappr.ipynb 71
def setup_trace_logging(
    report_id: str, # The report identifier
    verbosity: int = cfg.verbosity # The verbosity level
    ):
    "Setup logging for trace analysis"
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f'{report_id}_{timestamp}.jsonl'
    file_handler = logging.FileHandler(traces_dir / filename, mode='w')
    setup_logger('trace.file', file_handler)    
    console_handler = logging.StreamHandler()
    setup_logger('trace.console', console_handler, verbosity=verbosity)

# %% ../nbs/06_mappr.ipynb 72
def _build_base_data(
    event: str, # The event to log
    report_id: str, # The report identifier
    stage: Stage = None, # The stage of the pipeline
    framework_info: FrameworkInfo = None, # The framework information
    **extra_data: dict # Additional keyword arguments
    ):
    "Build base data dictionary for logging"
    base_data = {
        "timestamp": datetime.now().isoformat(),
        "event": event,
        "report_id": report_id,
    }
    
    if stage:
        base_data["stage"] = str(stage)
    
    if framework_info:
        base_data.update({
            "framework": str(framework_info.name),
            "framework_category": str(framework_info.category),
            "framework_theme_id": str(framework_info.theme_id),
        })
    
    base_data.update(extra_data)
    return base_data

# %% ../nbs/06_mappr.ipynb 73
def _format_console_msg(
    base_data: dict, # The base data to format
    verbosity: int, # The verbosity level
    stage: Stage, # The stage of the pipeline
    framework_info: FrameworkInfo # The framework information
    ):
    "Format console message based on verbosity level"
    report_id = base_data['report_id']
    event = base_data['event']
    
    if verbosity == 1:
        return f"{report_id}" + (f" - {stage}" if stage else "")
    elif verbosity == 2:
        parts = [report_id]
        if stage: parts.append(str(stage))
        parts.append(event)
        if framework_info:
            parts.append(f"{framework_info.name}/{framework_info.category}/{framework_info.theme_id}")
        return " - ".join(parts)
    else:  # verbosity == 3
        return json.dumps(base_data, indent=2)

# %% ../nbs/06_mappr.ipynb 74
def log_analysis_event(
    event: str, # The event to log
    report_id: str, # The report identifier
    stage: Stage = None, # The stage of the pipeline
    framework_info: FrameworkInfo = None, # The framework information
    **extra_data: dict # Additional keyword arguments
    ):
    "Log an analysis event to file and console with different verbosity levels"
    file_logger = logging.getLogger('trace.file')
    console_logger = logging.getLogger('trace.console')
    
    base_data = _build_base_data(event, report_id, stage, framework_info, **extra_data)
    
    # File logger - always full JSON
    file_logger.info(json.dumps(base_data, indent=2))
    
    # Console logger - verbosity-based
    if hasattr(console_logger, 'verbosity'):
        console_msg = _format_console_msg(base_data, console_logger.verbosity, 
                                         stage, framework_info)
        console_logger.info(console_msg)


# %% ../nbs/06_mappr.ipynb 76
def get_from_cache(
    table, # The cache table
    pk_value: str # The primary key value
    ):
    "Generic cache retrieval, returns None if not found"
    try:
        return table.get(pk_value)
    except NotFoundError:
        return None

# %% ../nbs/06_mappr.ipynb 77
def store_in_cache(
    table, # The cache table
    data: dict # The data to store
    ):
    "Generic cache storage with automatic timestamp"
    data['timestamp'] = datetime.now().isoformat()
    table.upsert(data)

# %% ../nbs/06_mappr.ipynb 78
async def limit(
    semaphore, # The semaphore to use
    coro, # The coroutine to execute
    delay: float = None # The delay to wait
    ):
    "Execute coroutine with semaphore concurrency control"
    async with semaphore:
        result = await coro
        if delay: await sleep(delay)
        return result

# %% ../nbs/06_mappr.ipynb 79
class TaggingResult(AttrDict):
    "The result of the theme tagging"
    def __init__(self, response: dict, framework_info: FrameworkInfo):
        self.is_core = response['is_core']
        self.reasoning = response['reasoning']
        self.confidence = response['confidence']
        self.framework_name = framework_info.name
        self.framework_category = framework_info.category
        self.framework_theme_id = framework_info.theme_id

# %% ../nbs/06_mappr.ipynb 80
class PipelineResults(dict):
    "The results of the pipeline"
    def __init__(self):
        super().__init__()
        self[Stage.STAGE1] = defaultdict(lambda: defaultdict(dict))
        self[Stage.STAGE2] = defaultdict(lambda: defaultdict(dict))
        self[Stage.STAGE3] = defaultdict(lambda: defaultdict(dict))

# %% ../nbs/06_mappr.ipynb 81
@patch
def __call__(
    self:PipelineResults, 
    stage: Stage = Stage.STAGE1, # The stage of the pipeline
    filter_type: str = "all" # The filter type
    ):
    "The results of the pipeline"
    themes = []
    for frameworks in self[stage].values():
        for categories in frameworks.values():
            for theme in categories.values():
                if filter_type == "all" or \
                   (filter_type == "tagged" and theme.is_core) or \
                   (filter_type == "untagged" and not theme.is_core):
                    themes.append(theme)
    return themes

# %% ../nbs/06_mappr.ipynb 82
cfg.pipeline = AttrDict({
    'select_section_prompt': select_section_sp,
    'tagging_prompt': tagging_sp_no_citation,
    'model': 'claude-sonnet-4-5',
    'cache_system': True,
    'cache_theme': True,
    'response_format': TagResult,
    'verbosity': 2,
    'call_delay': 1,
    'force_refresh': AttrDict({
        'sections': False,
        'stage1': False,
        'stage2': False,
        'stage3': False
    })
})

# %% ../nbs/06_mappr.ipynb 83
class PipelineOrchestrator:
    "The orchestrator of the pipeline"
    def __init__(self, 
                 report_id: str,  # The report identifier
                 hdgs: dict,  # The heading dictionary
                 eval_data: EvalData,  # The evaluation data
                 cfg: AttrDict  # The configuration
                 ):
        self.cfg_p = cfg.pipeline
        store_attr()
        setup_trace_logging(report_id, self.cfg_p.verbosity)
        self.results = PipelineResults()
        self.core_sections = None
        self.doc_content = None

# %% ../nbs/06_mappr.ipynb 84
@patch
async def identify_sections(self:PipelineOrchestrator, semaphore):
    "Identify core sections and log the selection"
    if not self.cfg_p.force_refresh.sections:
        cached = get_from_cache(sections_cache, self.report_id)
        if cached:
            self.core_sections = json.loads(cached.sections_selected)
            self.doc_content = extract_core_content(self.core_sections, self.hdgs)
            log_analysis_event(
                "sections_retrieved_from_cache",
                self.report_id,
                sections_selected=self.core_sections)
            return
    
    core_sections_result = await limit(
        semaphore,
        identify_core_sections(self.hdgs, self.cfg_p.select_section_prompt, self.cfg_p.model),
        self.cfg_p.call_delay
    )
    
    self.core_sections = core_sections_result['section_names']
    self.doc_content = extract_core_content(self.core_sections, self.hdgs)
    
    store_in_cache(sections_cache, {
        'report_id': self.report_id,
        'sections_selected': json.dumps(self.core_sections),
        'reasoning': core_sections_result['reasoning']
    })
    
    log_analysis_event(
        "sections_identified",
        self.report_id,
        sections_selected=self.core_sections,
        reasoning=core_sections_result['reasoning']
    )

# %% ../nbs/06_mappr.ipynb 85
@patch
def _tag_kwargs(self:PipelineOrchestrator):
    return {
        'system_prompt': self.cfg_p.tagging_prompt,
        'response_format': self.cfg_p.response_format,
        'model': self.cfg_p.model,
        'cache_system': self.cfg_p.cache_system,
        'cache_theme': self.cfg_p.cache_theme
    }

# %% ../nbs/06_mappr.ipynb 86
@patch
async def process_themes_batch(self:PipelineOrchestrator, 
                               themes, 
                               semaphore, 
                               stage, 
                               force_refresh,
                               log_fn=None
                               ):
    "Process multiple themes in parallel with rate limiting and caching"
    
    async def process_one(theme, framework_info):
        # Check cache first (unless force_refresh)
        if not force_refresh:
            pk = (self.report_id, str(stage), str(framework_info.name), 
                  str(framework_info.category), str(framework_info.theme_id))
            cached = get_from_cache(theme_cache, pk)
            if cached:
                result = TaggingResult(
                    {'is_core': cached.is_core, 'reasoning': cached.reasoning, 
                     'confidence': cached.confidence}, 
                    framework_info
                )
                if log_fn: 
                    log_analysis_event("theme_retrieved_from_cache", self.report_id, 
                                     stage=stage, framework_info=framework_info)
                return result
        
        # Cache miss - call LLM
        response = await tag_theme(
            doc_content=self.doc_content,
            theme=theme,
            **self._tag_kwargs()
        )
        parsed = parse_response(response)
        result = TaggingResult(parsed, framework_info)
        
        # Store in cache
        store_in_cache(theme_cache, {
            'report_id': self.report_id,
            'stage': str(stage),
            'framework': str(framework_info.name),
            'framework_category': str(framework_info.category),
            'framework_theme_id': str(framework_info.theme_id),
            'is_core': result.is_core,
            'reasoning': result.reasoning,
            'confidence': result.confidence
        })
        
        if log_fn: log_fn(result, framework_info)
        return result
    
    # Process with rate limiting
    results = [await limit(semaphore, process_one(*themes[0]), self.cfg_p.call_delay)]
    remaining = await gather(*[limit(semaphore, process_one(*t), self.cfg_p.call_delay) for t in themes[1:]])
    results.extend(remaining)
    
    return results


# %% ../nbs/06_mappr.ipynb 87
@patch
async def run_stage1(self:PipelineOrchestrator, semaphore):
    themes = []
    
    for item in self.eval_data.srf_enablers:
        framework_info = FrameworkInfo(Framework.SRF, FrameworkCat.ENABLERS, item.id)
        themes.append((format_enabler_theme(item), framework_info))
    
    for item in self.eval_data.srf_crosscutting_priorities:
        framework_info = FrameworkInfo(Framework.SRF, FrameworkCat.CROSSCUT, item.id)
        themes.append((format_crosscutting_theme(item), framework_info))
    
    log_fn = lambda result, finfo: log_analysis_event(
        "theme_tagged",
        self.report_id,
        stage=Stage.STAGE1,
        framework_info=finfo,
        is_core=result.is_core,
        reasoning=result.reasoning,
        confidence=result.confidence
    )

    results = await self.process_themes_batch(
        themes=themes, 
        semaphore=semaphore, 
        stage=Stage.STAGE1, 
        force_refresh=self.cfg_p.force_refresh.stage1,
        log_fn=log_fn
    )
    
    for result in results:
        self.results[Stage.STAGE1][result.framework_name][result.framework_category][result.framework_theme_id] = result

# %% ../nbs/06_mappr.ipynb 90
@patch
def get_stage1_context(self:PipelineOrchestrator) -> str:
    "Get formatted context from Stage 1 tagged themes"
    tagged_themes = self.results(Stage.STAGE1, filter_type="tagged")
    if not tagged_themes: 
        return ""
    
    context_parts = []
    for theme in tagged_themes:
        if theme.framework_category == str(FrameworkCat.ENABLERS):
            theme_data = next(t for t in self.eval_data.srf_enablers 
                            if t.id == theme.framework_theme_id)
        elif theme.framework_category == str(FrameworkCat.CROSSCUT):
            theme_data = next(t for t in self.eval_data.srf_crosscutting_priorities 
                            if t.id == theme.framework_theme_id)
        
        context_parts.append(f"- **{theme.framework_category} {theme_data.id}**: {theme_data.title}")
    
    return f"### Report Preliminary Context\nThis evaluation report covers the following Strategic Results Framework themes:\n" + "\n".join(context_parts)


# %% ../nbs/06_mappr.ipynb 93
@patch
async def run_stage2(self:PipelineOrchestrator, semaphore):
    "Run stage 2 - GCM objectives analysis with Stage 1 context"
    stage1_context = self.get_stage1_context()
    themes = []
    
    for gcm_obj in self.eval_data.gcm_objectives_small:
        framework_info = FrameworkInfo(Framework.GCM, FrameworkCat.OBJS, gcm_obj["id"])
        theme = format_gcm_theme(gcm_obj) + "\n\n" + stage1_context
        themes.append((theme, framework_info))
    
    log_fn = lambda result, finfo: log_analysis_event(
        "theme_tagged",
        self.report_id,
        stage=Stage.STAGE2,
        framework_info=finfo,
        is_core=result.is_core,
        reasoning=result.reasoning,
        confidence=result.confidence
    )

    results = await self.process_themes_batch(
        themes=themes,
        semaphore=semaphore,
        stage=Stage.STAGE2,
        force_refresh=self.cfg_p.force_refresh.stage2,
        log_fn=log_fn
    )
    
    for result in results: 
        self.results[Stage.STAGE2][result.framework_name][result.framework_category][result.framework_theme_id] = result


# %% ../nbs/06_mappr.ipynb 99
def get_filtered_srf_output_ids(
    results: PipelineResults, # PipelineResults
    eval_data: EvalData # EvalData
    ) -> list: # list of SRF output IDs
    "Get filtered SRF output IDs based on covered GCM themes."
    covered_gcm = results(Stage.STAGE2, filter_type="tagged")
    srf_output_ids = set()
    
    for gcm_theme in covered_gcm:
        gcm_id = gcm_theme.framework_theme_id
        if gcm_id in eval_data.gcm_srf_lut:
            srf_output_ids.update(eval_data.gcm_srf_lut[gcm_id])
    
    return list(srf_output_ids)

# %% ../nbs/06_mappr.ipynb 102
@patch
def get_combined_context(self:PipelineOrchestrator) -> str:
    "Get combined context from Stage 1 and Stage 2 tagged themes"
    stage1_context = self.get_stage1_context()
    tagged_gcm = self.results(Stage.STAGE2, filter_type="tagged")
    
    if not tagged_gcm:
        return stage1_context
    
    gcm_context = "\n".join([
        f"- **GCM {theme.framework_theme_id}**: {self.eval_data.gcm_objectives_small[int(theme.framework_theme_id)-1]['title']}" 
        for theme in tagged_gcm
    ])
    
    return f"{stage1_context}\n\n### Covered GCM Objectives\n{gcm_context}"

# %% ../nbs/06_mappr.ipynb 105
@patch
def get_filtered_srf_outputs(self:PipelineOrchestrator) -> list:
    "Get filtered SRF output IDs based on tagged GCM themes"
    tagged_gcm = self.results(Stage.STAGE2, filter_type="tagged")
    srf_output_ids = set()
    
    for gcm_theme in tagged_gcm:
        gcm_id = gcm_theme.framework_theme_id
        if gcm_id in self.eval_data.gcm_srf_lut:
            srf_output_ids.update(self.eval_data.gcm_srf_lut[gcm_id])
    
    return list(srf_output_ids)

# %% ../nbs/06_mappr.ipynb 107
@patch
async def run_stage3(self:PipelineOrchestrator, semaphore):
    "Run stage 3 - Targeted SRF outputs analysis with combined context"
    combined_context = self.get_combined_context()
    filtered_output_ids = self.get_filtered_srf_outputs()
    themes = []
    
    for output_id in filtered_output_ids:
        output_context = find_srf_output_by_id(self.eval_data, output_id)
        if output_context:
            framework_info = FrameworkInfo(Framework.SRF, FrameworkCat.OUTPUTS, output_id)
            theme = format_srf_output(output_context) + "\n\n" + combined_context
            themes.append((theme, framework_info))
    
    log_fn = lambda result, finfo: log_analysis_event(
        "theme_tagged",
        self.report_id,
        stage=Stage.STAGE3,
        framework_info=finfo,
        is_core=result.is_core,
        reasoning=result.reasoning,
        confidence=result.confidence
    )

    results = await self.process_themes_batch(
        themes=themes,
        semaphore=semaphore,
        stage=Stage.STAGE3,
        force_refresh=self.cfg_p.force_refresh.stage3,
        log_fn=log_fn
    )
    
    for result in results: 
        self.results[Stage.STAGE3][result.framework_name][result.framework_category][result.framework_theme_id] = result


# %% ../nbs/06_mappr.ipynb 111
def find_enriched_path(eval_id: str, md_dir: str):
    "Find the enriched markdown directory for an evaluation"
    eval_path = Path(md_dir) / eval_id
    if not eval_path.exists():
        raise FileNotFoundError(f"Evaluation directory not found: {eval_path}")
    
    report_dirs = eval_path.ls().filter(lambda d: d.is_dir() and d.name != 'enriched')
    if not report_dirs:
        raise FileNotFoundError(f"No report directory found in {eval_path}")
    
    doc_path = report_dirs[0] / 'enriched'
    if not doc_path.exists():
        raise FileNotFoundError(f"Enriched directory not found: {doc_path}")
    
    return doc_path

# %% ../nbs/06_mappr.ipynb 112
def parse_force_refresh(force_refresh_str: str, working_cfg):
    "Parse and apply force_refresh parameter to config"
    if force_refresh_str:
        refresh_items = [s.strip() for s in force_refresh_str.split(',')]
        for item in refresh_items:
            if item == 'sections':
                working_cfg.pipeline.force_refresh.sections = True
            elif item in ['stage1', 'stage2', 'stage3']:
                working_cfg.pipeline.force_refresh[item] = True

# %% ../nbs/06_mappr.ipynb 113
async def run_selected_stages(orchestrator, semaphore, stages_to_run):
    "Run only the selected pipeline stages"
    await orchestrator.identify_sections(semaphore)
    if 1 in stages_to_run:
        await orchestrator.run_stage1(semaphore)
    if 2 in stages_to_run:
        await orchestrator.run_stage2(semaphore)
    if 3 in stages_to_run:
        await orchestrator.run_stage3(semaphore)

# %% ../nbs/06_mappr.ipynb 114
@call_parse
def tag_evaluation(
    eval_id: str,  # Evaluation ID to process
    md_dir: str = "_data/md_library",  # Markdown directory
    stages: str = "1,2,3",  # Stages to run (comma-separated: 1,2,3)
    force_refresh: str = None  # Force refresh stages (comma-separated: sections,stage1,stage2,stage3)
):
    "Tag evaluation report against frameworks"
    # Use module's default config
    working_cfg = deepcopy(cfg)
    
    # Find enriched markdown path
    try:
        doc_path = find_enriched_path(eval_id, md_dir)
    except FileNotFoundError as e:
        logging.error(str(e))
        return
    
    # Load report and create heading structure
    report = load_report(doc_path)
    hdgs = create_heading_dict(report)
    
    # Parse parameters
    stages_to_run = [int(s.strip()) for s in stages.split(',')]
    parse_force_refresh(force_refresh, working_cfg)
    
    # Run pipeline
    orchestrator = PipelineOrchestrator(
        report_id=eval_id,
        hdgs=hdgs,
        eval_data=IOMEvalData(),
        cfg=working_cfg
    )
    
    semaphore = Semaphore(working_cfg.semaphore)
    asyncio.run(run_selected_stages(orchestrator, semaphore, stages_to_run))
    
    logging.info(f"Completed tagging for {eval_id}")

You might be interested also to look at the mappr module, because in our situation, we don't need the Chat object but we can use litellm "completion" or its async version directly as it is not a multiturn chat

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Ah, excellent point! Looking at your `mappr` module, I can see you're using `acompletion` directly for single-turn tasks like in `tag_theme`. 

For your heading fixing task, you could follow a similar pattern. Here's a small example:

```python
from litellm import completion

response = completion(
    model=cfg.lm,
    messages=[{"role": "user", "content": f"Fix these headings:\n{formatted_hdgs}"}],
    response_format=HeadingResult  # or your Pydantic model
)
```

A few questions:
1. Do you want to use the synchronous `completion` or async `acompletion`?
2. Should we create a Pydantic model for the response format (like your `CoreSectionsOutput` or `TagResult`)?
3. Would you like me to show you how to adapt your existing `fix_md` function to use this approach?

In this module, I'd like to reproduce the "enrich" module but using lisette (or rather 'completion' from litellm available in it). I've also provided the "mappr" module where I show how to use "completion" or "acompletion". But I'd like also to reconsider "enrich" and potentially simplify it. So let's reflect a bit, plan and approach the problem step by step.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Great! Let's think through this systematically. Looking at your `enrich` module, I see it does two main things:

1. **Fix heading hierarchy** - uses dspy to analyze and correct markdown headings
2. **Enrich images** - classifies image relevance, then describes relevant ones

Let me ask a few questions to help us plan:

**About the overall approach:**
- Do you want to keep both functions (heading fixing + image enrichment) or focus on one first?
- The current approach uses dspy for structured outputs. Do you want to switch entirely to using `completion` with Pydantic `response_format` (like in mappr)?

**About simplification opportunities:**
- The image enrichment currently has multiple steps (find refs → classify → describe → replace). Do you see any steps that could be combined or streamlined?
- The heading fixing applies corrections page-by-page. Would a different structure be simpler?

What aspects feel most complex or unnecessary in the current implementation?

- I'd like to tackle both headings fixing and image enrichment (description) using litellm/lisette
- for the image enrichment, I think we can do it in one go. The classif. step is an additional complexity that's not worth given the cost of an llm query for it.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good decisions! Let's plan this out:

**For heading fixing:**
- Use `completion` with a Pydantic model (similar to your `CoreSectionsOutput` in mappr)
- Keep the page-by-page correction approach or would you prefer to process all headings at once?

**For image enrichment:**
- Skip the classification step entirely
- Go directly: find image refs → describe all images → replace in markdown
- This means we'd describe every image we find, which is simpler

A few quick questions:

1. For headings: Should we process all pages' headings in one LLM call, or keep the current approach of analyzing them together but applying corrections page-by-page?

2. For images: Do you want to keep the context extraction (surrounding text) when describing images, or simplify further?

3. Should we create a new module file or would you like to modify the existing code in place?

- for headings, we could potentially process all pages headings in one LLM call. But I also like the ability to append page number to each headings (though it might be implemented in a simpler way)
- Not sure we should keep the context around the image link (again premature optimization maybe)

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good thinking on both points!

**For headings:**
We could format input as: `"# Introduction .... page 1"` (heading text + page marker), then ask the LLM to preserve those page markers in the output. That way we know where to apply each correction. Would that work?

**For images:**
I agree - let's start simple. Just pass the image itself to the vision model with a basic prompt like "Describe this figure for a technical report." We can always add context later if needed.

So our simplified flow would be:
1. **Headings**: Get all headings with page markers → one LLM call → apply corrections per page
2. **Images**: Find image refs → describe each image (vision model) → replace inline

Does this sound like the right level of simplicity? Should we start with headings or images first?

Let's approach the "headings" part step-by-step. What would be the next smallest step to make?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good! Let's start with the data structure. 

Looking at your current code, `get_hdgs_with_pages()` returns a list of dicts like:
```python
[{'heading': '# Introduction', 'page': 1}, ...]
```

For the LLM input, we need to format this. Should we:

**Option A:** Format as the LLM will see it: `"# Introduction .... page 1"`

**Option B:** Keep the structured dict and format it inside the LLM call function

Which feels cleaner to you? (Option A means one formatting function, Option B keeps data structured longer)

Looks fine. Just questioning the rules such as "- # for top-level (title or section 1, 2, 3...)". I don't agree, top-level section 1, 2, 3, should be "##" under the main title "#". But we don't want to hardcode/enforce it. We should only fix the original one, don't change all hierarchy. For instance in the example above, this is correct to keep '## 1. Introduction', '## 2. Related Work', ... 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Ah, you're absolutely right! The goal is to **fix errors** in the hierarchy, not impose a new structure.

So the rules should be more like:
- **Don't jump levels**: Can't go from `#` to `####` (like the Abstract example)
- **Respect numbering**: If something is numbered `1.1`, it should be one level deeper than `1.`
- **Preserve what works**: If `## 1. Introduction` and `## 2. Related Work` are consistent, keep them

So we're looking for inconsistencies to fix, not rewriting everything. Does that match what you're thinking?

Should we refine the prompt rules to focus on "fixing breaks" rather than "applying a hierarchy"?

Yes refine the prompt.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Here's a refined prompt focused on **fixing breaks** rather than imposing structure:

```python
prompt = """Fix markdown heading hierarchy errors while preserving the document's intended structure.

RULES - Only fix these errors:
1. **Level jumps**: Headings can only increase by one # at a time
   - Wrong: # Title → #### Abstract
   - Fixed: # Title → ## Abstract

2. **Numbering inconsistency**: Subsection numbers must be one level deeper
   - If you see: ## 3. Section → ## 3.1 Subsection
   - Fixed to: ## 3. Section → ### 3.1 Subsection

3. **Preserve page markers**: Keep the ".... page N" exactly as provided

4. **Don't change working structure**: If sections are consistently marked (e.g., all main sections use ##), keep it that way

5. **Decreasing levels is OK**: ### 3.1 → ## 4. is valid (new main section)

OUTPUT: Return a Python dictionary mapping each original heading (with page marker) to its fixed version.
If no fix needed, map to itself.

Headings to fix:
{headings_list}
"""
```

Does this capture the "fix errors only" approach better? Any rules to add or clarify?

Few remarks:
1. This prompt should be applicable to any doc so take care of not "overfitting" the example provided. Focus on the principles
2. What do you mean at point 3

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good catches!

**1. Making it more general:**
You're right - I'll remove specific section names and focus on principles only.

**2. About point 3 - "Preserve page markers":**
I was referring to the format from `format_hdgs()` which adds `".... page N"` to each heading. The LLM should keep that exact marker in the output so we know which page to apply corrections to.

But wait - looking at your code again, is this the right approach? Or should we parse the page info separately and not include it in the LLM prompt at all?

What do you think makes more sense?

About point 3, no, don't consider page number, the input should be something alike:
```
'# Deep Residual Learning for Image Recognition ',
 '#### Abstract',
 '## 1. Introduction',
 '## 2. Related Work',
 '## 3. Deep Residual Learning',
 '### 3.1. Residual Learning',
 '### 3.2. Identity Mapping by Shortcuts',
 '### 3.3. Network Architectures',
 '### 3.4. Implementation',
 '## 4. Experiments',
 '### 4.1. ImageNet Classification',
 '### 4.2. CIFAR-10 and Analysis',
 '### 4.3. Object Detection on PASCAL and MS COCO',
 '## References',
 '## A. Object Detection Baselines',
 '## PASCAL VOC',
 '## MS COCO',
 '## B. Object Detection Improvements',
 '## MS COCO',
 '## PASCAL VOC',
 '## ImageNet Detection',
 '## C. ImageNet Localization']
```

Or a modified version that could be more amenable to LLM

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Ah perfect! So we don't include page numbers in the LLM prompt at all. Much simpler.

So the flow would be:
1. Extract all headings from all pages (just the heading text)
2. Send to LLM → get back corrected headings
3. Apply corrections by matching original → corrected (using the page info we have separately)

For the input format to the LLM, should we:
- **Option A**: Send as a simple list (one heading per line)
- **Option B**: Number them for easier reference: `1. # Deep Residual...`
- **Option C**: Some other format?

Which would make it clearer for the LLM to process?

For the input format, what would best practice? One important point is that if we want a dict as output with {'original': 'fixed', ...}, we have to make sure that the original is strictly preserved (not risk of being altered by the LLM) to use it then to look it up and replace with the fixed one. Any best practice here?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Excellent question! That's a real risk - if the LLM slightly alters the original (extra space, etc.), the lookup breaks.

**Best practices to avoid this:**

**Option A - Use indices:**
- Input: Number each heading `1. # Deep Residual...`, `2. #### Abstract`, etc.
- Output: `{1: '# Deep Residual...', 2: '## Abstract', ...}` (index → corrected)
- We maintain the original list ourselves, only get corrections back

**Option B - Structured output (Pydantic):**
```python
class HeadingFix(BaseModel):
    index: int
    corrected: str
    changed: bool  # whether it needed fixing
```
- Send numbered list
- Get back list of fixes with index references

**Option C - Return only changes:**
- Send numbered list
- LLM returns only the ones that need fixing: `{2: '## Abstract', 16: '### PASCAL VOC', ...}`

Which approach feels most robust to you?

Let's try each approach. Let's start with option A

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great! Let's build Option A step by step.

First, let's create a function to format the headings with indices:

```python
def format_hdgs_indexed(hdgs: list[str]) -> str:
    return '\n'.join(f"{i}. {h}" for i, h in enumerate(hdgs))
```

This takes your list of headings and returns:
```
0. # Deep Residual Learning...
1. #### Abstract
2. ## 1. Introduction
...
```

Does this format look good, or would you prefer 1-indexed (starting from 1 instead of 0)?

In [ ]:
def fmt_hdgs_idx(hdgs: list[str]) -> str:
    return '\n'.join(f"{i}. {h}" for i, h in enumerate(hdgs))


In [ ]:
print(fmt_hdgs_idx(hdgs))

0. # Deep Residual Learning for Image Recognition 
1. #### Abstract
2. ## 1. Introduction
3. ## 2. Related Work
4. ## 3. Deep Residual Learning
5. ### 3.1. Residual Learning
6. ### 3.2. Identity Mapping by Shortcuts
7. ### 3.3. Network Architectures
8. ### 3.4. Implementation
9. ## 4. Experiments
10. ### 4.1. ImageNet Classification
11. ### 4.2. CIFAR-10 and Analysis
12. ### 4.3. Object Detection on PASCAL and MS COCO
13. ## References
14. ## A. Object Detection Baselines
15. ## PASCAL VOC
16. ## MS COCO
17. ## B. Object Detection Improvements
18. ## MS COCO
19. ## PASCAL VOC
20. ## ImageNet Detection
21. ## C. ImageNet Localization


Then we should update the prompt accordingly

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Perfect! Here's the updated prompt for the indexed format:

```python
prompt = """Fix markdown heading hierarchy errors while preserving the document's intended structure.

INPUT FORMAT: Each heading is prefixed with its index number (e.g., "0. # Title")

RULES - Only fix these errors:
1. **Level jumps**: Headings can only increase by one # at a time
   - Wrong: 0. # Title → 1. #### Abstract
   - Fixed: 0. # Title → 1. ## Abstract

2. **Numbering inconsistency**: Subsection numbers must be one level deeper
   - Wrong: 4. ## 3. Section → 5. ## 3.1 Subsection
   - Fixed: 4. ## 3. Section → 5. ### 3.1 Subsection

3. **Preserve working structure**: If sections are consistently marked, keep it

4. **Decreasing levels is OK**: Going from ### to ## is valid for new sections

OUTPUT: Return a Python dictionary mapping index to corrected heading (without the index prefix).
Only include entries that need changes. Example: {1: '## Abstract', 15: '### PASCAL VOC'}

Headings to analyze:
{headings_list}
"""
```

Should we test this prompt with your example, or refine it further first?

In [ ]:
prompt = """Fix markdown heading hierarchy errors while preserving the document's intended structure.

INPUT FORMAT: Each heading is prefixed with its index number (e.g., "0. # Title")

RULES - Only fix these errors:
1. **Level jumps**: Headings can only increase by one # at a time
   - Wrong: 0. # Title → 1. #### Abstract
   - Fixed: 0. # Title → 1. ## Abstract

2. **Numbering inconsistency**: Subsection numbers must be one level deeper
   - Wrong: 4. ## 3. Section → 5. ## 3.1 Subsection
   - Fixed: 4. ## 3. Section → 5. ### 3.1 Subsection

3. **Preserve working structure**: If sections are consistently marked, keep it

4. **Decreasing levels is OK**: Going from ### to ## is valid for new sections

OUTPUT: Return a Python dictionary mapping index to corrected heading (without the index prefix).
Only include entries that need changes. Example: {{1: '## Abstract', 15: '### PASCAL VOC'}}

Headings to analyze:
{headings_list}
"""

In [ ]:
hdgs_fmt = fmt_hdgs_idx(hdgs)

Can we apply this prompt on $`hdgs_fmt` using lisette? You can check example on $`mappr` to see how to use "completion' for instance.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Sure! Here's how to call it with `completion`:

```python
from litellm import completion
import json

response = completion(
    model=cfg.lm,
    messages=[{"role": "user", "content": prompt.format(headings_list=hdgs_fmt)}],
    api_key=cfg.api_key
)

result = json.loads(response.choices[0].message.content)
```

Would you like to try running this and see what we get back?

In [ ]:
from litellm import completion
import json
import os

response = completion(
    model='gemini/gemini-2.0-flash',
    messages=[{"role": "user", "content": prompt.format(headings_list=hdgs_fmt)}],
    api_key=os.getenv('GEMINI_API_KEY')
)

result = json.loads(response.choices[0].message.content)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

What's the problem above.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The LLM returned something that's not valid JSON. Let's check what it actually returned:

```python
print(response.choices[0].message.content)
```

This will show us what the model gave back. It might have wrapped the JSON in markdown code blocks or added explanation text.

Want to run that to see what we got?

In [ ]:
print(response.choices[0].message.content)

```python
def fix_headings(headings):
  """
  Fixes markdown heading hierarchy errors while preserving the document's intended structure.

  Args:
    headings: A list of strings, where each string is a heading prefixed with its index number (e.g., "0. # Title").

  Returns:
    A dictionary mapping index to corrected heading (without the index prefix).
    Only include entries that need changes.
  """

  corrections = {}
  previous_level = 0

  for i, heading in enumerate(headings):
    index = int(heading.split('.')[0])
    text = heading.split('. ', 1)[1]
    level = text.count('#')
    content = text.split('# ', 1)[1] if '#' in text else text # Extract the heading text after removing the '#'

    # Rule 1: Level jumps
    if i == 0:
      previous_level = level
      continue

    if level > previous_level + 1:
      new_level = previous_level + 1
      new_heading = '#' * new_level + ' ' + content
      corrections[index] = new_heading
      level = new_level
    
    previous_lev

What do you think?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

The model wrote Python code instead of returning JSON! But interestingly, it got the right answer: `{1: '## Abstract'}` - fixing the jump from `#` to `####`.

Two issues:
1. It wrapped the response in markdown code blocks
2. It generated code rather than just the dictionary

We can fix this by using `response_format` with a Pydantic model (like in your mappr code). This forces structured output.

Should we switch to Option B (Pydantic approach) or try to improve the prompt to get pure JSON?

Yes let's try a pydantic approach

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great! Let's define a Pydantic model for the response. Here's a simple structure:

```python
from pydantic import BaseModel

class HeadingCorrections(BaseModel):
    corrections: dict[int, str]  # index → corrected heading
```

Then we'd call it like:

```python
response = completion(
    model='gemini/gemini-2.0-flash',
    messages=[{"role": "user", "content": prompt.format(headings_list=hdgs_fmt)}],
    response_format=HeadingCorrections,
    api_key=os.getenv('GEMINI_API_KEY')
)
```

Want to try this and see if it returns structured JSON?

In [ ]:
from pydantic import BaseModel

class HeadingCorrections(BaseModel):
    corrections: dict[int, str]  # index → corrected heading

In [ ]:
response = completion(
    model='gemini/gemini-2.0-flash',
    messages=[{"role": "user", "content": prompt.format(headings_list=hdgs_fmt)}],
    response_format=HeadingCorrections,
    api_key=os.getenv('GEMINI_API_KEY')
)


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



BadRequestError: litellm.BadRequestError: VertexAIException BadRequestError - {
  "error": {
    "code": 400,
    "message": "* GenerateContentRequest.generation_config.response_schema.properties[\"corrections\"].properties: should be non-empty for OBJECT type\n",
    "status": "INVALID_ARGUMENT"
  }
}


What do you think?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

The Gemini API doesn't like `dict[int, str]` as a response format - it needs explicit properties. 

Two options:

**Option 1 - Use a list instead:**
```python
class HeadingFix(BaseModel):
    index: int
    corrected: str

class HeadingCorrections(BaseModel):
    corrections: list[HeadingFix]
```

**Option 2 - Try Claude/Anthropic** which handles dict types better in structured output.

Which approach would you prefer?

Let's try claude/anthropic

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good choice! Let's try with Claude. You'll need to update the model and use your Anthropic API key:

```python
response = completion(
    model='claude-sonnet-4-5',
    messages=[{"role": "user", "content": prompt.format(headings_list=hdgs_fmt)}],
    response_format=HeadingCorrections,
    api_key=os.getenv('ANTHROPIC_API_KEY')
)
```

Want to try running this and see if it returns the structured dict?

In [ ]:
from lisette.core import completion
import os

response = completion(
    model='claude-sonnet-4-5',
    messages=[{"role": "user", "content": prompt.format(headings_list=hdgs_fmt)}],
    response_format=HeadingCorrections,
    api_key=os.getenv('ANTHROPIC_API_KEY')
)

In [ ]:
print(response.choices[0].message.content)

{"corrections": {"1": "## Abstract", "15": "### PASCAL VOC", "16": "### MS COCO", "18": "### MS COCO", "19": "### PASCAL VOC", "20": "### ImageNet Detection", "21": "### C. ImageNet Localization"}}


In [ ]:
hdgs

['# Deep Residual Learning for Image Recognition ',
 '#### Abstract',
 '## 1. Introduction',
 '## 2. Related Work',
 '## 3. Deep Residual Learning',
 '### 3.1. Residual Learning',
 '### 3.2. Identity Mapping by Shortcuts',
 '### 3.3. Network Architectures',
 '### 3.4. Implementation',
 '## 4. Experiments',
 '### 4.1. ImageNet Classification',
 '### 4.2. CIFAR-10 and Analysis',
 '### 4.3. Object Detection on PASCAL and MS COCO',
 '## References',
 '## A. Object Detection Baselines',
 '## PASCAL VOC',
 '## MS COCO',
 '## B. Object Detection Improvements',
 '## MS COCO',
 '## PASCAL VOC',
 '## ImageNet Detection',
 '## C. ImageNet Localization']

In [ ]:
pg_nb = 1
p = read_pgs('files/test/md_all/resnet', join=False)[pg_nb-1]
p[:100]

TypeError: read_pgs() got an unexpected keyword argument 'join'

In [ ]:
import json

result = json.loads(response.choices[0].message.content)
corrections = result['corrections']

fixed_hdgs = hdgs.copy()
for idx_str, corrected in corrections.items():
    idx = int(idx_str)
    fixed_hdgs[idx] = corrected

# Show the result
for i, h in enumerate(fixed_hdgs):
    print(f"{i}. {h}")

0. # Deep Residual Learning for Image Recognition 
1. ## Abstract
2. ## 1. Introduction
3. ## 2. Related Work
4. ## 3. Deep Residual Learning
5. ### 3.1. Residual Learning
6. ### 3.2. Identity Mapping by Shortcuts
7. ### 3.3. Network Architectures
8. ### 3.4. Implementation
9. ## 4. Experiments
10. ### 4.1. ImageNet Classification
11. ### 4.2. CIFAR-10 and Analysis
12. ### 4.3. Object Detection on PASCAL and MS COCO
13. ## References
14. ## A. Object Detection Baselines
15. ### PASCAL VOC
16. ### MS COCO
17. ## B. Object Detection Improvements
18. ### MS COCO
19. ### PASCAL VOC
20. ### ImageNet Detection
21. ### C. ImageNet Localization
